# Baseball Awards — Visual Analysis

This notebook runs the six historical awards plus the sabermetrics leaderboards against the local Postgres warehouse and renders one chart per award.

**Prereqs (one-time):**
1. `make up && make load && make migrate` from the repo root.
2. `make notebook` to open this notebook against the containerized Postgres.

All connection info is pulled from environment variables set by `docker-compose.yml`.

In [ ]:
import os
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)

DSN = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER','baseball')}:"
    f"{os.getenv('POSTGRES_PASSWORD','baseball')}@"
    f"{os.getenv('POSTGRES_HOST','postgres')}:5432/"
    f"{os.getenv('POSTGRES_DB','baseball')}"
)
engine = create_engine(DSN)
print('Connected to', DSN.split('@')[-1])

## 1. Heaviest Hitters — Top 20 historical roster weights

In [ ]:
df = pd.read_sql("""
    SELECT p.team_id, p.year_id, t.name AS team_name, p.avg_weight_lbs
    FROM   analytics.mv_team_physiques p
    LEFT   JOIN raw.teams t USING (team_id, year_id)
    ORDER  BY p.avg_weight_lbs DESC NULLS LAST
    LIMIT  20
""", engine)
df['label'] = df['team_name'].fillna(df['team_id']) + ' (' + df['year_id'].astype(str) + ')'

ax = sns.barplot(data=df, y='label', x='avg_weight_lbs', palette='rocket_r')
ax.set_title('Heaviest Rosters — Top 20')
ax.set_xlabel('Avg batter weight (lbs)')
ax.set_ylabel('')
ax.set_xlim(df['avg_weight_lbs'].min() - 5, df['avg_weight_lbs'].max() + 2)
plt.tight_layout(); plt.show()

## 2. Shortest Sluggers — Top 20 ascending

In [ ]:
df = pd.read_sql("""
    SELECT p.team_id, p.year_id, t.name AS team_name, p.avg_height_in
    FROM   analytics.mv_team_physiques p
    LEFT   JOIN raw.teams t USING (team_id, year_id)
    ORDER  BY p.avg_height_in ASC NULLS LAST
    LIMIT  20
""", engine)
df['label'] = df['team_name'].fillna(df['team_id']) + ' (' + df['year_id'].astype(str) + ')'

ax = sns.barplot(data=df, y='label', x='avg_height_in', palette='mako')
ax.set_title('Shortest Rosters — Top 20')
ax.set_xlabel('Avg batter height (in)')
plt.tight_layout(); plt.show()

## 3. Biggest Spenders — Payroll by year

In [ ]:
df = pd.read_sql("""
    SELECT year_id, MAX(total_payroll_usd) AS max_payroll
    FROM   analytics.mv_team_payroll
    GROUP  BY year_id
    ORDER  BY year_id
""", engine)
ax = sns.lineplot(data=df, x='year_id', y='max_payroll', marker='o', color='crimson')
ax.set_title('Maximum single-team payroll by year')
ax.set_xlabel('Year'); ax.set_ylabel('Max payroll (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
plt.tight_layout(); plt.show()

## 4. Cost per Win — 2010 leaderboard

In [ ]:
df = pd.read_sql("""
    SELECT t.name AS team_name, t.w AS wins, mp.total_payroll_usd,
           ROUND(mp.total_payroll_usd::numeric / NULLIF(t.w,0), 2) AS cost_per_win
    FROM   analytics.mv_team_payroll mp
    JOIN   raw.teams t USING (team_id, year_id)
    WHERE  mp.year_id = 2010 AND t.year_id = 2010 AND t.w > 0
    ORDER  BY cost_per_win ASC
""", engine)
df['cost_per_win_m'] = df['cost_per_win'] / 1e6
ax = sns.barplot(data=df, y='team_name', x='cost_per_win_m', palette='viridis')
ax.set_title('2010 Cost Per Win — every MLB team'); ax.set_xlabel('USD per win (millions)'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 5. Priciest Starter — Top 15 single-season \$ per start

In [ ]:
df = pd.read_sql("""
    SELECT pe.full_name AS pitcher, pi.year_id, pi.team_id,
           pi.gs, s.salary, ROUND(s.salary::numeric / pi.gs, 2) AS cost_per_start
    FROM   raw.pitching pi
    JOIN   raw.salaries s USING (player_id, year_id, team_id)
    JOIN   raw.people pe USING (player_id)
    WHERE  pi.gs >= 10
    ORDER  BY cost_per_start DESC NULLS LAST
    LIMIT  15
""", engine)
df['label'] = df['pitcher'] + ' (' + df['year_id'].astype(str) + ')'
df['cps_m']  = df['cost_per_start'] / 1e6
ax = sns.barplot(data=df, y='label', x='cps_m', palette='flare')
ax.set_title('Priciest single-season starting pitchers'); ax.set_xlabel('USD per start (millions)'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 6. Canadian Ace — Best ERA by Toronto/Montreal pitchers

In [ ]:
df = pd.read_sql("""
    SELECT pe.full_name AS pitcher, pi.year_id, pi.team_id, pi.era, pi.innings_pitched
    FROM   raw.pitching pi
    JOIN   raw.people pe USING (player_id)
    WHERE  pi.team_id IN ('TOR','MON') AND pi.era > 0 AND pi.ipouts >= 162
    ORDER  BY pi.era ASC, pi.ipouts DESC
    LIMIT  15
""", engine)
df['label'] = df['pitcher'] + ' (' + df['team_id'] + ' ' + df['year_id'].astype(str) + ')'
ax = sns.barplot(data=df, y='label', x='era', palette='crest')
ax.set_title('Canadian Ace — Lowest qualified ERA (TOR + MON)'); ax.set_xlabel('ERA'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## Bonus — Era-adjusted ERA over time

Demonstrates the PRD's "structural shift variations" callout: a 2.50 ERA in 1968 and a 2.50 ERA in 2000 are not comparable. The z-score axis controls for league baseline.

In [ ]:
df = pd.read_sql("""
    SELECT year_id, AVG(league_era_mean) AS lg_era
    FROM   analytics.mv_league_pitching_baseline
    GROUP  BY year_id ORDER BY year_id
""", engine)
ax = sns.lineplot(data=df, x='year_id', y='lg_era', color='steelblue')
ax.axhline(df['lg_era'].mean(), ls='--', color='gray', alpha=0.7)
ax.set_title('League-average ERA over time (era effects)'); ax.set_xlabel('Year'); ax.set_ylabel('Avg league ERA')
plt.tight_layout(); plt.show()